# Radar interferometry (InSAR)

The velocity field of an ice sheet is the quantity a diagnostic model most needs to be judged against, and the technique that maps it over whole continents is radar interferometry. A synthetic-aperture radar carried on a satellite illuminates the surface with microwaves and records not only the strength of the echo but its phase, the position within the wave cycle at which the echo returns. The phase advances by a full cycle for every half wavelength of change in the distance to the ground, and the wavelength is centimetres, so comparing the phase of two radar images of the same scene measures changes in range to a small fraction of a wavelength. Interferometric synthetic-aperture radar, or InSAR, turns that exquisite sensitivity into maps of how the ice moves and where it goes afloat. This chapter develops the method; it draws on the propagation physics of {doc}`../radar/em-waves` and complements the optical feature tracking of {doc}`panchromatic-imagery-lab`.


## Phase and baselines

Form the difference in phase between two radar images of the same area and the result, the interferogram, is a pattern of fringes. Those fringes contain two things mixed together. One is topography: if the two images were taken from slightly separated orbits, the parallax between them encodes surface elevation, the principle behind radar digital elevation models. The other is motion: if the surface moved between the two acquisitions, the line-of-sight component of that displacement shifts the phase directly. Separating the two is the central task. A known elevation model can be used to remove the topographic phase and leave the motion, or a third image can be combined so that the topographic contribution cancels. What remains, after the orbital and topographic terms are flattened away, is the displacement of the ice along the radar's line of sight over the time between images.


## From phase to a velocity vector

Several steps turn that raw phase into a usable velocity. The two images must be co-registered to a fraction of a pixel, the interferogram formed and flattened, and then the phase, which is only known modulo one cycle, must be unwrapped into a continuous field by counting fringes outward from a reference. The unwrapped phase is scaled to displacement and geocoded onto the map. Because a single interferogram measures only the one-dimensional projection of the motion onto the line of sight, recovering the full horizontal velocity vector requires combining passes flown in different directions, the ascending and descending orbits, or fusing interferometry with the offset tracking that works where the phase has decorrelated. Over fast, crevassed ice where the surface changes too much between passes for the phase to stay coherent, that speckle tracking, the radar analogue of optical feature tracking, takes over. A data-processing lab for this chapter forms a small interferogram, unwraps it, and converts the phase to a velocity.


## Glaciological applications

Interferometry produced the first detailed measurement of an ice stream in motion, on Rutford Ice Stream in 1993 {cite}`goldstein1993`, and it has since grown into continent-wide velocity maps of Antarctica and Greenland assembled from years of satellite passes {cite}`mouginot2019`. These maps are the data that diagnostic models are tuned to, and the basal friction of {doc}`../thermomechanics/basal-motion` is inferred by adjusting a model until its surface velocity matches them. A second, equally important use exploits the sensitivity of the phase to the small vertical motion of floating ice flexing under the ocean tide. Differencing interferograms taken at different tidal stages reveals the grounding line, the boundary between grounded and floating ice, with a precision no other method matches, and tracking that line through time has documented the retreat of the grounding lines of Pine Island and Thwaites that signals the marine instability of {doc}`../ice_flow/flow-approximations`. Velocity over the whole ice sheet and the position of the grounding line are two of the most consequential observations in the field, and both come from the phase of a radar echo.

A single interferogram built from two satellite passes, each from roughly 700 km altitude, contains hundreds of millions of independent phase measurements, each sensitive to millimetre-to-centimetre surface displacements along the line of sight. The tidal-flexure signal is perhaps the sharpest demonstration of that sensitivity: fringes marking the rise and fall of floating ice under the ocean tide appear in ice that is 80 km or more from the nearest open water, the phase responding to a vertical motion of centimetres propagating inland through a sheet whose surface gives no other visible sign of it.


---


## Lab: ice velocity from InSAR
```{note}
The sections above develops the phase equation, the unwrapping problem, and the connection to continental velocity mosaics; this lab makes those things run on real data.
```
A synthetic-aperture radar measures how far away the ground is, to a fraction of a centimetre, by recording the phase of the echo. Repeat the measurement twelve days later and subtract: anything that moved shows up as a fringe pattern in the phase difference. That pattern is an interferogram, and the fringes are the ice moving.
By the end of this lab you should be able to
- locate Sentinel-1 single-look complex (SLC) scenes at the Alaska Satellite Facility and understand what the data volumes actually mean,
- submit an on-demand interferogram job through ASF's HyP3 service, or load a pre-formed product to skip the compute wait,
- load wrapped phase and coherence, plot the fringe pattern, and read fringe spacing as range change,
- unwrap a 1-D profile by cumulative phase integration, then unwrap a small 2-D patch with `scikit-image`, and identify where coherence breaks unwrapping,
- convert unwrapped phase to line-of-sight displacement, divide by the time baseline to get velocity, and project to horizontal with the look geometry,
- mask low-coherence pixels and explain what tides do to the phase of a floating shelf.
The target is a well-studied West Antarctic ice stream where the chapter's expected velocity magnitudes are tens to hundreds of metres per year. That makes the fringe density tractable and the numbers checkable.

## Data sources and volumes
The European Space Agency's Sentinel-1 satellites have been imaging the polar regions on 6- and 12-day repeat cycles since 2014. The raw archive is held at the **Alaska Satellite Facility** (ASF), `https://search.asf.alaska.edu`, and the Python client `asf_search` lets you query it programmatically.
Before you write a single line of code, read this paragraph carefully, because the numbers in it will shape every decision you make.
A single Sentinel-1 SLC granule covering one swath pass is **4–8 GB** on disk. Forming one interferogram requires two SLCs (the reference and secondary acquisitions) plus a DEM to remove topographic phase (another few hundred MB) plus precise orbit files. The processing pipeline — co-registration, interferogram formation, adaptive filtering, geocoding — is a multi-hour job on a workstation with tens of GB of RAM. **This is genuinely the part of glaciology where laptops lose.** It is not a question of patience; it is a question of whether the operating system will let the process allocate the memory the coregistration kernel needs.
ASF's **HyP3** service (Hybrid Pluggable Processing Pipeline, `https://hyp3-api.asf.alaska.edu`) exists precisely to move that compute burden off your machine. You submit a job that names two SLC granules, HyP3 forms the interferogram on cloud hardware overnight, and the next morning you download a ~100 MB GeoTIFF package containing the wrapped phase, coherence, amplitude, and ancillary layers — geocoded and ready for analysis. The rest of this lab works entirely with that ~100 MB product.
```{note}
Track A (cells below) searches SLCs with `asf_search` and submits a HyP3 job. It requires a free NASA Earthdata account and is **not executed at build time** — all cells are marked with `# verify:` comments so you can read the API calls without running them. Track B starts from a provided HyP3 product GeoTIFF and runs completely offline.
```

In [ ]:
# verify: Track A — SLC search with asf_search
# Requires: pip install asf_search hyp3_sdk
# Requires: a free NASA Earthdata account at https://urs.earthdata.nasa.gov
#
# The target is a Rutford Ice Stream descending pair from the same relative
# orbit, separated by exactly 12 days (one Sentinel-1 repeat cycle).
#
# import asf_search as asf
#
# results = asf.search(
#     platform=asf.SENTINEL1,
#     processingLevel=asf.SLC,
#     intersectsWith='POINT(-84.0 -78.0)',   # Rutford Ice Stream, Antarctica
#     start='2020-01-01',
#     end='2020-03-01',
#     beamMode='IW',
#     flightDirection='DESCENDING',
#     relativeOrbit=100,
# )
#
# for r in results[:4]:
#     print(r.properties['fileID'],
#           r.properties['startTime'],
#           f"{r.properties['sizeMB']:.0f} MB")
#
# -- Expected output: four or more scenes, each ~4000-7000 MB.
# -- Pick two 12-day-apart scenes as reference and secondary.
print('Track A: SLC search cell (not executed at build). Read the comments.')

In [ ]:
# verify: Track A — submit a HyP3 InSAR job
#
# import hyp3_sdk as hyp3
#
# api = hyp3.HyP3()   # prompts for Earthdata credentials on first run
#
# ref_granule = 'S1A_IW_SLC__1SDV_20200112T203045_20200112T203112_030770_038759_xxxx'
# sec_granule = 'S1A_IW_SLC__1SDV_20200124T203045_20200124T203112_030945_038E6F_xxxx'
#
# job = api.submit_insar_job(
#     granule1=ref_granule,
#     granule2=sec_granule,
#     looks='20x4',
#     include_dem=True,
#     include_look_vectors=True,
# )
# print('Job submitted:', job.job_id)
# print('Typical wall time: 2-4 hours on HyP3 cloud infrastructure.')
# print('Download when status == SUCCEEDED:')
# print('  api.download_files(job.files, location="./hyp3_output/")')
#
# The downloaded package contains (among others):
#   *_wrapped_phase.tif   — wrapped phase in radians, [-pi, pi]
#   *_coherence.tif       — coherence, [0, 1]
#   *_amplitude.tif       — backscatter intensity
#   *_look_vector_*.tif   — incidence angle and azimuth per pixel
print('Track A: HyP3 submission cell (not executed at build). Read the comments.')

## Loading a HyP3 product
The cell below is the entry point for everyone. It loads a pre-cropped GeoTIFF pair — wrapped phase and coherence — clipped to a ~50 km patch over the Rutford Ice Stream trunk. The crop keeps the file small enough (a few MB) to live in the book repository. Everything from here on runs without an Earthdata account or a HyP3 job.
The Rutford Ice Stream drains into the Ronne Ice Shelf and moves at roughly 300–400 m yr$^{-1}$ near its centre, making it one of the most thoroughly observed ice streams on Earth. The first satellite radar interferogram of an ice stream was formed over this glacier in 1993 {cite}`goldstein1993`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
# rasterio reads GeoTIFFs with geospatial metadata attached.
# If not installed: pip install rasterio
try:
    import rasterio
    from rasterio.transform import rowcol
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print('rasterio not found — falling back to synthetic data for plotting.')
# Expected file locations after downloading the book's data bundle.
DATA_DIR = Path('../../data/insar')   # adjust if your layout differs
PHASE_FILE = DATA_DIR / 'rutford_wrapped_phase.tif'
COH_FILE   = DATA_DIR / 'rutford_coherence.tif'
def load_band(path):
    """Return (array, transform, crs) from a single-band GeoTIFF."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        arr[arr == src.nodata] = np.nan
        return arr, src.transform, src.crs
if HAS_RASTERIO and PHASE_FILE.exists():
    phase_wrapped, transform, crs = load_band(PHASE_FILE)
    coherence, _, _               = load_band(COH_FILE)
    nrows, ncols = phase_wrapped.shape
    print(f'Loaded phase array: {nrows} x {ncols} pixels')
    print(f'Coherence range: {np.nanmin(coherence):.2f} \u2013 {np.nanmax(coherence):.2f}')
else:
    # Synthetic stand-in: a plane of fringes with lower coherence near the edges.
    print('Data files not found. Generating synthetic interferogram for demonstration.')
    nrows, ncols = 300, 400
    y, x = np.mgrid[0:nrows, 0:ncols].astype(float)
    # Linear range-change gradient mimicking ~350 m/yr motion at 12-day baseline
    # Sentinel-1 C-band wavelength 0.0555 m, incidence ~35 deg
    # velocity 350 m/yr -> LOS ~0.97 m / 12 days -> phase ~35 rad over scene
    phase_wrapped = np.mod(
        (35.0 / ncols) * x + (8.0 / nrows) * y + np.pi, 2 * np.pi
    ) - np.pi
    coherence = np.clip(
        0.85 - 0.4 * ((x - ncols/2)**2 / (ncols/2)**2 +
                       (y - nrows/2)**2 / (nrows/2)**2),
        0.0, 1.0
    ).astype(np.float32)
    transform = None
    print(f'Synthetic array: {nrows} x {ncols} pixels')

In [ ]:
# A cyclic colormap makes the 2π phase wrapping visually natural.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(phase_wrapped, cmap='hsv', vmin=-np.pi, vmax=np.pi,
                     origin='upper', interpolation='nearest')
fig.colorbar(im0, ax=axes[0], label='wrapped phase (rad)')
axes[0].set_title('Wrapped interferometric phase')
axes[0].set_xlabel('range (pixels)')
axes[0].set_ylabel('azimuth (pixels)')
im1 = axes[1].imshow(coherence, cmap='gray', vmin=0, vmax=1,
                     origin='upper', interpolation='nearest')
fig.colorbar(im1, ax=axes[1], label='coherence')
axes[1].set_title('Coherence')
axes[1].set_xlabel('range (pixels)')
plt.tight_layout()
plt.show()
# Count fringes across the scene centre (one fringe = one full 2pi cycle).
centre_row = nrows // 2
phase_row = phase_wrapped[centre_row, :]
# Each sign flip from +pi to -pi marks a fringe boundary.
transitions = np.where(np.diff(np.sign(phase_row)))[0]
# Transitions come in pairs (ascending and descending zero crossings).
approx_fringes = len(transitions) // 2
print(f'Approximate fringe count across centre row: {approx_fringes}')

## Fringes and displacement
The chapter derives the central equation of InSAR. After removing topographic and orbital phase, what remains is
$$\phi = \frac{4\pi}{\lambda}\,\Delta r,$$
where $\lambda$ is the radar wavelength, $\Delta r$ is the change in slant range between acquisitions, and $\phi$ is the interferometric phase in radians. One complete fringe — a $2\pi$ phase cycle — corresponds to exactly half a wavelength of range change:
$$\Delta r_{\text{fringe}} = \frac{\lambda}{2}.$$
Sentinel-1 operates in C-band at 5.405 GHz, giving $\lambda \approx 0.0555$ m. One fringe therefore encodes 2.77 cm of range change. If you count ten fringes across a 20 km swath, the surface moved roughly 28 cm toward or away from the satellite during the 12-day interval, which works out to about 8.5 m yr$^{-1}$ along the line of sight — sensible for a slow mountain glacier but far too small for Rutford.
The cell below computes the total range change implied by the fringe count you just measured, and checks it against an expected velocity range.

In [ ]:
LAMBDA = 0.0555        # Sentinel-1 C-band wavelength, m
TIME_BASELINE = 12.0   # days between acquisitions
PIXEL_SPACING_M = 80.0 # approximate ground range pixel spacing after multilook, m
# Range change per fringe
dr_per_fringe = LAMBDA / 2.0
print(f'Range change per fringe: {dr_per_fringe * 100:.2f} cm')
# Total range change implied by the fringe count across the scene
scene_width_m = ncols * PIXEL_SPACING_M
total_dr_m = approx_fringes * dr_per_fringe
print(f'Scene width: {scene_width_m / 1e3:.0f} km')
print(f'Total range change across scene: {total_dr_m:.2f} m')
# Line-of-sight velocity (m/yr) from range change over the time baseline
los_velocity = total_dr_m / (TIME_BASELINE / 365.25)
print(f'Implied LOS velocity: {los_velocity:.0f} m/yr')
# Project to horizontal using incidence angle theta_i
# For Sentinel-1 IW mode over Antarctica: theta_i ~ 35 degrees
# The LOS unit vector has a range component cos(theta_i) for a flat surface.
THETA_INC = np.radians(35.0)
horiz_velocity = los_velocity / np.cos(THETA_INC)
print(f'Projected horizontal velocity: {horiz_velocity:.0f} m/yr')
print()
print('Expected for Rutford Ice Stream trunk: 300\u2013400 m/yr ({cite}`mouginot2019`)')

## Phase unwrapping
The wrapped phase is only known modulo $2\pi$. To get a continuous displacement field you need to add the right integer multiple of $2\pi$ to each pixel — a process called phase unwrapping. The cleanest way to think about it is this: if you walk from a reference pixel to any other pixel by a path along which the phase never jumps by more than $\pi$ in a single step, you can integrate the phase gradient along the path and recover the absolute phase. That works perfectly when the surface is coherent everywhere. It fails at shear margins, where crevassing destroys coherence between acquisitions, and in summer melt zones, where liquid water changes the dielectric constant between passes. Those are exactly the places you most want to know the velocity.
### 1-D profile unwrapping
The simplest unwrapper operates on a single row. Walk from left to right; whenever the phase jumps by more than $\pi$ between adjacent pixels, you have crossed a fringe boundary and you add or subtract $2\pi$ to bring the running total back into a continuous range. This is `numpy.unwrap`.

In [ ]:
# 1-D unwrap along the centre row
centre_row = nrows // 2
phase_1d = phase_wrapped[centre_row, :].copy()
coh_1d   = coherence[centre_row, :]
# Mask pixels with coherence below a threshold before unwrapping.
COH_THRESHOLD = 0.35
mask_1d = coh_1d < COH_THRESHOLD
phase_1d_masked = phase_1d.copy()
phase_1d_masked[mask_1d] = np.nan
# Fill NaNs with linear interpolation so numpy.unwrap has a continuous signal.
x_all = np.arange(ncols)
good = np.isfinite(phase_1d_masked)
if good.sum() > 2:
    phase_filled = np.where(good, phase_1d, np.interp(x_all, x_all[good], phase_1d[good]))
else:
    phase_filled = phase_1d.copy()
phase_unwrapped_1d = np.unwrap(phase_filled)
# Re-apply the coherence mask so low-coherence pixels show as NaN in the output.
phase_unwrapped_1d[mask_1d] = np.nan
# Convert unwrapped phase to range change and then to LOS displacement.
dr_1d = phase_unwrapped_1d * LAMBDA / (4 * np.pi)  # m of range change
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(phase_1d, lw=0.8, color='steelblue')
axes[0].set_ylabel('wrapped phase (rad)')
axes[0].set_title(f'Centre row (row {centre_row}) — phase profile')
axes[0].axhline( np.pi, color='gray', ls=':', lw=0.8)
axes[0].axhline(-np.pi, color='gray', ls=':', lw=0.8)
axes[1].plot(coh_1d, lw=0.8, color='darkorange')
axes[1].axhline(COH_THRESHOLD, color='red', ls='--', lw=1, label=f'threshold = {COH_THRESHOLD}')
axes[1].set_ylabel('coherence')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[2].plot(dr_1d, lw=0.8, color='forestgreen')
axes[2].set_ylabel('range change (m)')
axes[2].set_xlabel('range pixel')
plt.tight_layout()
plt.show()
total_range_change = np.nanmax(dr_1d) - np.nanmin(dr_1d)
print(f'Total range change along centre row: {total_range_change:.3f} m')

### 2-D patch unwrapping with scikit-image
The 1-D unwrapper is fragile: a single corrupted pixel breaks every estimate downstream of it. Real InSAR processors use 2-D algorithms — minimum-cost-flow or least-squares — that can route the integration path around incoherent regions. `scikit-image`'s `restoration.unwrap_phase` implements a fast 2-D unwrapper based on the Flynn minimum-discontinuity algorithm.
The unwrapper still struggles where coherence drops below about 0.3. In the shear margins of an ice stream, where the surface is a chaos of crevasses, coherence is essentially zero and the unwrapper has no signal to work with. That is the zone where speckle tracking — cross-correlating amplitude patches rather than differencing phase — takes over, as the chapter describes.

In [ ]:
try:
    from skimage.restoration import unwrap_phase as ski_unwrap
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print('scikit-image not found: pip install scikit-image')
# Work on a central sub-patch to keep memory and time manageable.
r0, r1 = nrows // 4, 3 * nrows // 4
c0, c1 = ncols // 4, 3 * ncols // 4
patch_phase = phase_wrapped[r0:r1, c0:c1].copy()
patch_coh   = coherence[r0:r1, c0:c1].copy()
# Mask low-coherence pixels (set to 0 so phase is undefined).
COH_MASK_2D = 0.30
patch_phase[patch_coh < COH_MASK_2D] = 0.0
if HAS_SKIMAGE:
    phase_uw_2d = ski_unwrap(patch_phase)
    # Re-apply the mask to the unwrapped output.
    phase_uw_2d[patch_coh < COH_MASK_2D] = np.nan
    dr_2d = phase_uw_2d * LAMBDA / (4 * np.pi)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    im0 = axes[0].imshow(patch_phase, cmap='hsv', vmin=-np.pi, vmax=np.pi,
                         origin='upper')
    fig.colorbar(im0, ax=axes[0], label='phase (rad)')
    axes[0].set_title('Wrapped phase (patch)')
    im1 = axes[1].imshow(patch_coh, cmap='gray', vmin=0, vmax=1,
                         origin='upper')
    fig.colorbar(im1, ax=axes[1], label='coherence')
    axes[1].set_title(f'Coherence (masked < {COH_MASK_2D})')
    im2 = axes[2].imshow(dr_2d, cmap='RdBu_r', origin='upper')
    fig.colorbar(im2, ax=axes[2], label='range change (m)')
    axes[2].set_title('Unwrapped range change')
    plt.tight_layout()
    plt.show()
    print(f'Patch shape: {patch_phase.shape}')
    print(f'Fraction of pixels surviving coherence mask: '
          f'{(patch_coh >= COH_MASK_2D).mean():.2f}')
    print(f'Range change span: {np.nanmin(dr_2d):.3f} to {np.nanmax(dr_2d):.3f} m')
else:
    print('Skipping 2-D unwrap: scikit-image not available.')

## Phase to velocity
The unwrapped range change $\Delta r$ accumulated over the time baseline $\Delta t$ gives a line-of-sight velocity
$$v_{\text{LOS}} = \frac{\Delta r}{\Delta t}.$$
The radar looks from the side, not straight down. The angle between the line of sight and the vertical is the incidence angle $\theta_i$. For a surface moving purely horizontally (a reasonable first approximation for ice flowing on a nearly flat bed far from the margins), the LOS component picks up only the horizontal motion projected onto the range direction:
$$v_{\text{LOS}} = v_h \sin\theta_i \cos\alpha,$$
where $\alpha$ is the azimuth angle between the satellite's flight direction and the ice flow direction. The full inversion for $v_h$ therefore requires knowing the flow azimuth, which you either get from a prior velocity map or by combining ascending and descending passes {cite}`mouginot2019`. Here we assume flow is perpendicular to the orbit (maximising the LOS projection, so $\cos\alpha \approx 1$) and use a nominal incidence angle of 35°.

In [ ]:
# Convert the 2-D range change to horizontal velocity.
# If scikit-image was not available, fall back to the 1-D row result.
dt_yr = TIME_BASELINE / 365.25   # time baseline in years
if HAS_SKIMAGE:
    # LOS velocity field, m/yr
    v_los = dr_2d / dt_yr
    # Project to horizontal: v_h = v_los / (sin(theta_i) * cos(alpha))
    # With alpha = 0 (flow in range direction) and theta_i = 35 deg:
    v_horiz = v_los / np.sin(THETA_INC)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    vmax = np.nanpercentile(np.abs(v_horiz), 98)
    im0 = axes[0].imshow(v_los, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
    fig.colorbar(im0, ax=axes[0], label='LOS velocity (m/yr)')
    axes[0].set_title('Line-of-sight velocity')
    im1 = axes[1].imshow(v_horiz, cmap='viridis', vmin=0, vmax=vmax, origin='upper')
    fig.colorbar(im1, ax=axes[1], label='horizontal velocity (m/yr)')
    axes[1].set_title(f'Horizontal velocity (incidence = {np.degrees(THETA_INC):.0f}°)')
    plt.tight_layout()
    plt.show()
    median_v = np.nanmedian(v_horiz)
    print(f'Median horizontal velocity over patch: {median_v:.0f} m/yr')
    print(f'Expected for Rutford trunk: ~300\u2013400 m/yr ({chr(123)}cite{chr(125)}`mouginot2019`)')
else:
    # Fall back: use the 1-D result
    v_los_1d = dr_1d / dt_yr
    v_horiz_1d = v_los_1d / np.sin(THETA_INC)
    print(f'1-D centre-row velocity range: '
          f'{np.nanmin(v_horiz_1d):.0f} to {np.nanmax(v_horiz_1d):.0f} m/yr')

## Tasks
**Task 1 — coherence masking threshold.** Return to the `COH_MASK_2D` variable in the 2-D unwrapping cell and try values of 0.1, 0.2, 0.3, 0.5, and 0.7. For each threshold, record the fraction of pixels that survive the mask and the range of the resulting velocity field. Plot both quantities as a function of threshold and identify the knee where you start discarding valid ice-covered pixels. What does this tell you about choosing a masking strategy in a region with heterogeneous surface conditions?
**Task 2 — tides on a floating shelf.** A 12-day interferogram spanning two different tidal states will have a phase contribution from vertical tide in addition to horizontal flow. For the Ronne Ice Shelf, tidal amplitudes reach 1–2 m. Using the phase equation $\phi = 4\pi\Delta r / \lambda$ and a purely vertical displacement $\Delta z$ with $\Delta r \approx \Delta z \cos\theta_i$:
- compute the number of fringes a 1 m tidal difference adds to the interferogram at $\theta_i = 35°$,
- explain in one paragraph why tidal fringes are concentric around the grounding line rather than parallel to the ice-flow direction,
- describe how differencing two interferograms at different tidal phases isolates the grounding line position, and connect this to the ApRES basal melt measurement in {doc}`../radar/apres`.
```python
# Your calculation for Task 2 here.
lambda_c = 0.0555   # m, Sentinel-1 C-band
theta_i  = np.radians(35.0)
delta_z  = 1.0      # m, tidal amplitude
# delta_r = delta_z * cos(theta_i)  (vertical displacement maps to range)
# phi     = 4 * pi * delta_r / lambda_c
# fringes = phi / (2 * pi)
```
**Task 3 — the 1-D unwrapping failure mode.** Find or construct a 1-D phase profile that has a coherence dropout in the middle (set 20 consecutive pixels to NaN). Run the interpolation-then-unwrap procedure from Phase unwrapping on it. Then shift the dropout window to a fringe boundary and run again. Describe in two or three sentences why the second case fails and the first may not, and what that implies for unwrapping across shear margins.

In [ ]:
# Task 2 calculation: tidal fringes on a floating shelf
lambda_c = 0.0555    # m, Sentinel-1 C-band
theta_i  = np.radians(35.0)
for delta_z in [0.5, 1.0, 2.0]:
    delta_r = delta_z * np.cos(theta_i)
    phi     = 4 * np.pi * delta_r / lambda_c
    fringes = phi / (2 * np.pi)
    print(f'Tidal amplitude {delta_z:.1f} m -> '
          f'range change {delta_r*100:.1f} cm -> '
          f'{fringes:.1f} additional fringes')

## Synthesis
A single 12-day interferogram gives you velocity over one swath, at one moment in time, from one look direction. The continental velocity mosaics that the diagnostic modelling chapters depend on — the maps that constrain basal friction fields across the whole Amundsen sector — are assembled from thousands of such interferograms, merged with speckle-tracking results where the surface is too chaotic for phase coherence, and averaged over years to suppress tidal and seasonal noise {cite}`mouginot2019`.
Two important limits of interferometry are worth carrying forward.
**Where coherence dies, speckle tracking takes over.** Crevassed shear margins, summer ablation surfaces, and fast-moving calving fronts all decorrelate within 6–12 days. There the phase carries no information about displacement, but the amplitude image still has a recognisable speckle pattern. Cross-correlating amplitude patches between acquisitions — the radar analogue of the optical feature tracking in {doc}`panchromatic-imagery-lab` — recovers offsets at metre-scale precision rather than centimetre-scale, but it works where InSAR cannot. The Rutford velocity map produced by Wuite et al. {cite}`wuite2015` blends both methods.
**The phase and the grounding line.** The tidal task above sketches how tidal flexure in an interferogram traces the grounding line. Repeated interferograms from different tidal phases can pin the grounding line to within tens of metres, far better than any optical method, and tracking that line through time has documented the retreat of the Pine Island and Thwaites grounding lines that signals the marine ice-sheet instability discussed in {doc}`../ice_flow/flow-approximations` {cite}`joughin2014`.
**Write a short synthesis** (one to two paragraphs) addressing the questions below.
1. Your velocity estimate relied on a known incidence angle, a known flow azimuth, and the assumption of purely horizontal motion. Which of these assumptions introduces the largest error for an ice stream near a grounding line, and why? How does adding a descending-orbit interferogram help?
2. The fringe density in your interferogram is roughly uniform across the trunk, then compresses near the margins before coherence breaks down entirely. Explain what the margin fringe compression tells you physically, and why it is precisely those shear margins that both models and observers most need accurate velocity for.
3. The chapter notes that the velocity mosaics used to initialise prognostic ice-sheet models come from a particular era of satellite coverage. If you were designing a future observing system, what repeat interval and look-geometry combination would best constrain both the velocity field and the grounding-line position simultaneously?

In [ ]:
# Convenience summary table — reproduces the key numbers from the lab
# in one place so the synthesis questions have numbers to point at.
print('=== Lab summary ===')
print(f'Sentinel-1 wavelength: {LAMBDA*100:.2f} cm')
print(f'Range change per fringe: {LAMBDA/2*100:.2f} cm')
print(f'Time baseline: {TIME_BASELINE:.0f} days = {dt_yr:.4f} yr')
print(f'Incidence angle (nominal): {np.degrees(THETA_INC):.0f} deg')
print()
# Tidal fringe summary
delta_z_typical = 1.0   # m, typical Ronne Shelf tidal amplitude
delta_r_tide = delta_z_typical * np.cos(THETA_INC)
fringes_tide = (4 * np.pi * delta_r_tide / LAMBDA) / (2 * np.pi)
print(f'Tidal fringe count for {delta_z_typical:.0f} m tidal amplitude: {fringes_tide:.1f}')
print()
# Data-volume summary for Track A context
slc_size_gb = (4, 8)    # typical range for one Sentinel-1 SLC
hyp3_size_mb = 100      # approximate HyP3 product size
print(f'Data volumes:')
print(f'  One SLC: {slc_size_gb[0]}\u2013{slc_size_gb[1]} GB')
print(f'  Two SLCs + DEM + orbits: ~{2*slc_size_gb[0]+1}\u2013{2*slc_size_gb[1]+1} GB raw')
print(f'  HyP3 on-demand product: ~{hyp3_size_mb} MB')
print(f'  Compression ratio: ~{(2*slc_size_gb[0])*1000 // hyp3_size_mb}\u2013'
      f'{(2*slc_size_gb[1])*1000 // hyp3_size_mb}x')